# Jaffle Shop: From Raw Data to Semantic Layer

This notebook demonstrates how to go from **raw synthetic data** to a fully auto-ingested **SLayer semantic layer** backed by DuckDB.

We'll:
1. Generate 3 years of realistic Jaffle Shop data using `jafgen`
2. Load it into DuckDB with proper schemas and foreign keys
3. Auto-ingest the database with SLayer, which discovers tables, relationships, and builds rollup joins
4. Explore the resulting models and run queries through the semantic layer

**Prerequisites:** `pip install motley-slayer[examples]`

In [ ]:
import os
import sys
import tempfile

import duckdb

# Add project root to path for local development
sys.path.insert(0, os.path.join(os.getcwd(), "..", ".."))

from jaffle_shop_duckdb import SCHEMA_FILE, create_schema, generate_data, load_data

from slayer.core.models import DatasourceConfig
from slayer.core.query import Field, SlayerQuery
from slayer.engine.ingestion import ingest_datasource
from slayer.engine.query_engine import SlayerQueryEngine
from slayer.storage.yaml_storage import YAMLStorage

## Step 1: Generate Data & Load into DuckDB

[jafgen](https://github.com/dbt-labs/jaffle-shop-generator) simulates a chain of Jaffle shops over time, producing realistic data with seasonality, customer personas, and store openings. We generate 3 years of data and load it into DuckDB.

The schema has 7 tables with foreign key relationships:

```
customers  <--  orders  -->  stores
                  |
            order_items  -->  products  <--  supplies
                  
customers  <--  tweets
```

In [ ]:
work_dir = tempfile.mkdtemp(prefix="jaffle_shop_")
db_path = os.path.join(work_dir, "jaffle_shop.duckdb")

# Generate CSV data (this takes ~45 seconds for 3 years)
data_dir = generate_data(work_dir, years=3)

# Create DuckDB database with schema and load data
conn = duckdb.connect(db_path)
create_schema(conn, SCHEMA_FILE)
load_data(conn, data_dir)

Let's peek at what we loaded:

In [ ]:
for table in ["customers", "stores", "products", "orders", "order_items", "supplies", "tweets"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:>15}: {count:>8,} rows")

print("\nSample orders:")
conn.execute("SELECT * FROM orders LIMIT 5").df()

In [ ]:
conn.close()

## Step 2: Auto-Ingest with SLayer

SLayer's `ingest_datasource()` introspects the database to discover:
- All tables and their column types
- Foreign key relationships between tables
- Transitive FK chains (e.g. `order_items` -> `orders` -> `customers`)

For each table it automatically generates:
- **Dimensions** from all columns (own + rolled-up from referenced tables)
- **Measures** like `count`, `sum`, `avg` for numeric columns
- **Rollup SQL** with LEFT JOINs for tables that reference others

In [ ]:
# Configure the datasource and storage
storage = YAMLStorage(base_dir=os.path.join(work_dir, "slayer_models"))
ds = DatasourceConfig(name="jaffle_shop", type="duckdb", database=db_path)
storage.save_datasource(ds)

# Auto-ingest all tables
models = ingest_datasource(datasource=ds)

# Set default time dimensions for time-series queries
for model in models:
    if model.name == "orders":
        model.default_time_dimension = "ordered_at"
    elif model.name == "tweets":
        model.default_time_dimension = "tweeted_at"
    storage.save_model(model)

print(f"Discovered {len(models)} models:")
for m in sorted(models, key=lambda m: m.name):
    rollup = " (rollup)" if m.sql else ""
    print(f"  {m.name}: {len(m.dimensions)} dimensions, {len(m.measures)} measures{rollup}")

Let's inspect the **orders** model in detail. Notice how SLayer automatically rolled up dimensions from `customers` and `stores` via foreign keys:

In [ ]:
orders_model = next(m for m in models if m.name == "orders")

print("Dimensions:")
for dim in orders_model.dimensions:
    pk = " [PK]" if dim.primary_key else ""
    print(f"  {dim.name} ({dim.type}){pk}")

print(f"\nMeasures:")
for meas in orders_model.measures:
    sql_info = f" sql={meas.sql}" if meas.sql else ""
    print(f"  {meas.name} ({meas.type}){sql_info}")

print(f"\nGenerated SQL:\n{orders_model.sql}")

## Step 3: Query Through the Semantic Layer

Now we can query the data using SLayer's semantic API. Instead of writing SQL, we describe **what** we want: measures, dimensions, filters, and time granularity. SLayer generates and executes the SQL.

In [ ]:
engine = SlayerQueryEngine(storage=storage)

### Revenue by store

Using the auto-rolled-up `stores__name` dimension from the orders model:

In [ ]:
result = engine.execute(query=SlayerQuery(
    model="orders",
    fields=[Field(formula="count"), Field(formula="order_total_sum")],
    dimensions=[{"name": "stores__name"}],
    order=[{"column": {"name": "order_total_sum"}, "direction": "desc"}],
))

for row in result.data:
    store = row["orders.stores__name"]
    count = row["orders.count"]
    revenue = row["orders.order_total_sum"]
    print(f"  {store}: {count:,} orders, ${revenue:,.2f} revenue")

### Monthly order trends with cumulative sum

SLayer supports time dimensions with granularity, plus formula functions like `cumsum`:

In [ ]:
result = engine.execute(query=SlayerQuery(
    model="orders",
    time_dimensions=[{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    fields=[
        Field(formula="count"),
        Field(formula="cumsum(count)", name="cumulative"),
    ],
    order=[{"column": {"name": "ordered_at"}, "direction": "asc"}],
))

print(f"{'Month':<12} {'Orders':>8} {'Cumulative':>12}")
print("-" * 34)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    count = row["orders.count"]
    cumulative = row["orders.cumulative"]
    print(f"{month:<12} {count:>8,} {cumulative:>12,}")

### Top products by number of orders

This query traverses the `order_items` -> `products` rollup join automatically:

In [ ]:
result = engine.execute(query=SlayerQuery(
    model="order_items",
    fields=[Field(formula="count"), Field(formula="quantity_sum")],
    dimensions=[{"name": "products__name"}, {"name": "products__type"}],
    order=[{"column": {"name": "quantity_sum"}, "direction": "desc"}],
))

print(f"{'Product':<35} {'Type':<10} {'Line Items':>12} {'Qty Sold':>10}")
print("-" * 70)
for row in result.data:
    name = row["order_items.products__name"]
    ptype = row["order_items.products__type"]
    count = row["order_items.count"]
    qty = row["order_items.quantity_sum"]
    print(f"{name:<35} {ptype:<10} {count:>12,} {qty:>10,}")

### Month-over-month revenue change

SLayer's `change()` function computes the difference from the previous period using a self-join (no edge NULLs, gap-safe):

In [ ]:
result = engine.execute(query=SlayerQuery(
    model="orders",
    time_dimensions=[{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    fields=[
        Field(formula="order_total_sum"),
        Field(formula="change(order_total_sum)", name="mom_change"),
        Field(formula="change_pct(order_total_sum)", name="mom_pct"),
    ],
    order=[{"column": {"name": "ordered_at"}, "direction": "asc"}],
    limit=12,
))

print(f"{'Month':<12} {'Revenue':>12} {'MoM Change':>12} {'MoM %':>8}")
print("-" * 48)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    pct = row["orders.mom_pct"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    pct_str = f"{pct:+.1%}" if pct is not None else "-"
    print(f"{month:<12} ${rev:>11,.2f} {chg_str:>12} {pct_str:>8}")

## Summary

Starting from raw CSV data, we:
1. **Generated** 3 years of realistic e-commerce data with `jafgen`
2. **Loaded** it into DuckDB with proper schemas and foreign keys
3. **Auto-ingested** 7 tables into SLayer, which discovered all FK relationships and built rollup joins
4. **Queried** the semantic layer using high-level constructs (measures, dimensions, formulas) instead of raw SQL

SLayer handled the complexity of multi-table joins, time granularity, and analytical functions like cumulative sums and period-over-period comparisons -- all from a simple declarative query API.